# 01 - Dynamic Factor Model (DFM) Introduction

## Overview

The **Dynamic Factor Model** (DFM) extracts a small number of latent common
factors from a large panel of observed time series. It is widely used in
macroeconomic monitoring, nowcasting, and index construction.

### Model Equations

$$
\begin{align}
y_t &= \Lambda f_t + \varepsilon_t, \quad \varepsilon_t \sim N(0, R) \quad \text{(observation equation)} \\
f_t &= \Phi f_{t-1} + \eta_t, \quad \eta_t \sim N(0, I_K) \quad \text{(state/factor transition)}
\end{align}
$$

where:
- $y_t \in \mathbb{R}^N$ — vector of $N$ observed series at time $t$
- $f_t \in \mathbb{R}^K$ — vector of $K$ latent factors ($K \ll N$)
- $\Lambda \in \mathbb{R}^{N \times K}$ — factor loading matrix
- $\Phi \in \mathbb{R}^{K \times K}$ — factor autoregressive (VAR) dynamics
- $R = \text{diag}(\sigma_1^2, \ldots, \sigma_N^2)$ — idiosyncratic noise covariance

### Identification

The factors and loadings are only jointly identified up to rotation. We impose:
1. $Q = I_K$ (factor innovation variance fixed to identity)
2. Lower-triangular constraint on the first $K$ rows of $\Lambda$

### Estimation

We use **Maximum Likelihood** via the Kalman filter. The Kalman filter
computes the likelihood $p(y_{1:T} | \theta)$ exactly, and we optimize
$\theta = (\Lambda, \Phi, R)$ numerically. Initial values come from PCA.

### This Notebook

We apply the DFM to `us_macro_panel.csv` — a panel of 15 US macroeconomic
series (monthly, 2000–2023, standardized). We estimate models with $K=1$ and
$K=2$ factors, compare with PCA and statsmodels, and select the number of
factors via BIC.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from kalmanbox import DynamicFactorModel

import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.dpi': 100,
})

print('Imports OK')

In [ ]:
# Load US macro panel dataset
data_dir = Path(__file__).resolve().parent / 'data' if '__file__' in dir() else Path('data')
df = pd.read_csv(data_dir / 'us_macro_panel.csv', parse_dates=['date'])
df = df.set_index('date')

series_names = df.columns.tolist()
print(f'Shape: {df.shape}  ({df.shape[0]} months, {df.shape[1]} series)')
print(f'Period: {df.index[0].strftime("%Y-%m")} to {df.index[-1].strftime("%Y-%m")}')
print(f'\nSeries: {series_names}')
print(f'\nDescriptive statistics:\n{df.describe().round(3)}')

## Exploratory Analysis

Before fitting the DFM, we examine the correlation structure and apply PCA
to get a preliminary sense of how many common factors drive the panel.

In [ ]:
# Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Correlation matrix
corr = df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=axes[0], square=True,
            cbar_kws={'shrink': 0.8})
axes[0].set_title('Correlation Matrix of Macro Panel')

# PCA: eigenvalue decomposition of covariance
from sklearn.decomposition import PCA

y_arr = df.values
pca_full = PCA()
pca_full.fit(y_arr)

# Scree plot
axes[1].bar(range(1, len(pca_full.explained_variance_ratio_) + 1),
            pca_full.explained_variance_ratio_, alpha=0.7, color='steelblue',
            label='Individual')
axes[1].plot(range(1, len(pca_full.explained_variance_ratio_) + 1),
             np.cumsum(pca_full.explained_variance_ratio_), 'ro-',
             label='Cumulative')
axes[1].axhline(0.8, color='gray', linestyle='--', alpha=0.5, label='80% threshold')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Explained Variance Ratio')
axes[1].set_title('PCA Scree Plot')
axes[1].legend()
axes[1].set_xticks(range(1, len(pca_full.explained_variance_ratio_) + 1))

plt.tight_layout()
plt.show()

# Variance explained by first K components
for k in range(1, 6):
    cumvar = np.sum(pca_full.explained_variance_ratio_[:k])
    print(f'  PC 1-{k}: {cumvar:.1%} variance explained')

## DFM with K=1 Factor

We start with the simplest case: a single common factor driving all 15 series.
The state-space matrices are:

| Matrix | Dimension | Description |
|--------|-----------|-------------|
| $Z = \Lambda$ | $(15 \times 1)$ | Factor loadings |
| $T = \Phi$ | $(1 \times 1)$ | Factor AR(1) coefficient |
| $H = R$ | $(15 \times 15)$ | Diagonal idiosyncratic variances |
| $Q = I_1$ | $(1 \times 1)$ | Fixed to 1 for identification |

In [ ]:
# Fit DFM with K=1 factor
y = df.values.astype(np.float64)

model_k1 = DynamicFactorModel(y, k_factors=1, factor_order=1,
                               endog_names=series_names)
results_k1 = model_k1.fit(compute_se=False)

print(results_k1.summary())
print(f'\nFactor AR(1) coefficient (Phi): {results_k1.ssm.T[0, 0]:.4f}')

In [ ]:
# Extract the smoothed factor and plot against key macro indicators
factor_k1 = results_k1.smoothed_state[:, 0]  # shape (T,)
dates = df.index

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Factor vs real activity indicators
ax = axes[0]
ax.plot(dates, factor_k1, 'k-', linewidth=2, label='Factor 1 (DFM K=1)')
for s in ['gdp_growth', 'industrial_production', 'payrolls']:
    ax.plot(dates, df[s].values, '--', alpha=0.5, label=s)
ax.set_ylabel('Standardized Value')
ax.set_title('Extracted Factor vs Real Activity Indicators')
ax.legend(loc='upper right', fontsize=8)

# Factor vs financial indicators
ax = axes[1]
ax.plot(dates, factor_k1, 'k-', linewidth=2, label='Factor 1 (DFM K=1)')
for s in ['cpi_inflation', 'fed_funds_rate', 'sp500_returns']:
    ax.plot(dates, df[s].values, '--', alpha=0.5, label=s)
ax.set_ylabel('Standardized Value')
ax.set_title('Extracted Factor vs Financial/Price Indicators')
ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

## DFM with K=2 Factors

With two factors, we can potentially separate "real activity" from
"price/financial" dynamics. The identification constraint is a
lower-triangular structure on the first 2 rows of $\Lambda$:

$$
\Lambda = \begin{pmatrix}
\lambda_{11} & 0 \\
\lambda_{21} & \lambda_{22} \\
\lambda_{31} & \lambda_{32} \\
\vdots & \vdots \\
\lambda_{N1} & \lambda_{N2}
\end{pmatrix}
$$

In [ ]:
# Fit DFM with K=2 factors
model_k2 = DynamicFactorModel(y, k_factors=2, factor_order=1,
                               endog_names=series_names)
results_k2 = model_k2.fit(compute_se=False)

print(results_k2.summary())

# Factor transition matrix (Phi)
Phi = results_k2.ssm.T[:2, :2]
print(f'\nFactor transition matrix Phi:')
print(f'  {Phi}')
print(f'\nEigenvalues of Phi: {np.linalg.eigvals(Phi).round(4)}')

## Extracted Factors and Factor Loadings

We now visualize the two extracted factors and their loadings on each series.

In [ ]:
# Plot extracted factors (smoothed)
factors_k2 = results_k2.smoothed_state[:, :2]  # shape (T, 2)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(dates, factors_k2[:, 0], 'b-', linewidth=1.5, label='Factor 1')
axes[0].set_ylabel('Factor Value')
axes[0].set_title('Factor 1 (K=2 model)')
axes[0].legend()
axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)

axes[1].plot(dates, factors_k2[:, 1], 'r-', linewidth=1.5, label='Factor 2')
axes[1].set_ylabel('Factor Value')
axes[1].set_title('Factor 2 (K=2 model)')
axes[1].legend()
axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Factor loadings heatmap (Lambda)
Lambda = results_k2.ssm.Z[:, :2]  # shape (N, 2)

fig, ax = plt.subplots(figsize=(8, 10))
sns.heatmap(
    pd.DataFrame(Lambda, index=series_names, columns=['Factor 1', 'Factor 2']),
    annot=True, fmt='.3f', cmap='RdBu_r', center=0,
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Loading'},
)
ax.set_title('Factor Loadings Matrix $\\Lambda$ (K=2)')
ax.set_ylabel('Series')
plt.tight_layout()
plt.show()

# Print loadings table
print('\nFactor Loadings:')
loadings_df = pd.DataFrame(Lambda, index=series_names,
                            columns=['Factor 1', 'Factor 2'])
print(loadings_df.round(4).to_string())

## Selecting the Number of Factors

We compare models with $K = 1, 2, 3$ factors using the BIC (Bayesian
Information Criterion). Lower BIC indicates a better trade-off between
fit and parsimony. We also show a "scree plot" of eigenvalues from PCA
for comparison.

In [ ]:
# BIC-based selection of number of factors
k_range = [1, 2, 3]
bic_values = []
aic_values = []
loglik_values = []

results_dict = {1: results_k1, 2: results_k2}

for k in k_range:
    if k in results_dict:
        res = results_dict[k]
    else:
        model_k = DynamicFactorModel(y, k_factors=k, factor_order=1,
                                      endog_names=series_names)
        res = model_k.fit(compute_se=False)
        results_dict[k] = res
    bic_values.append(res.bic)
    aic_values.append(res.aic)
    loglik_values.append(res.loglike)
    print(f'  K={k}: LogL={res.loglike:>10.2f}  AIC={res.aic:>10.2f}  BIC={res.bic:>10.2f}  params={res.k_params}')

best_k_bic = k_range[np.argmin(bic_values)]
print(f'\nBest K by BIC: {best_k_bic}')

# Plot BIC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, bic_values, 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Number of Factors (K)')
axes[0].set_ylabel('BIC')
axes[0].set_title('BIC vs Number of Factors')
axes[0].set_xticks(k_range)
best_idx = np.argmin(bic_values)
axes[0].plot(k_range[best_idx], bic_values[best_idx], 'r*', markersize=20,
             label=f'Best K={best_k_bic}')
axes[0].legend()

# Eigenvalue scree plot for comparison
eigenvalues = pca_full.explained_variance_
axes[1].bar(range(1, len(eigenvalues) + 1), eigenvalues, alpha=0.7,
            color='steelblue')
axes[1].axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Kaiser criterion')
axes[1].set_xlabel('Component')
axes[1].set_ylabel('Eigenvalue')
axes[1].set_title('PCA Eigenvalues (Kaiser Rule)')
axes[1].legend()
axes[1].set_xticks(range(1, len(eigenvalues) + 1))

plt.tight_layout()
plt.show()

## Comparison with Static PCA

PCA extracts static factors (no dynamics) from the cross-sectional
covariance. The DFM extends this by modeling factor persistence via the
VAR transition equation and estimating the full likelihood. We compare
the two approaches.

In [ ]:
# Compare DFM factors with PCA factors
pca_2 = PCA(n_components=2)
pca_factors = pca_2.fit_transform(y_arr)

# DFM factors (K=2)
dfm_factors = results_k2.smoothed_state[:, :2]

# Correlations between DFM and PCA factors
# Note: sign/scale may differ, so we compare absolute correlations
corr_matrix = np.zeros((2, 2))
for i in range(2):
    for j in range(2):
        corr_matrix[i, j] = np.abs(np.corrcoef(dfm_factors[:, i], pca_factors[:, j])[0, 1])

print('Absolute correlations between DFM factors and PCA components:')
print(pd.DataFrame(corr_matrix, index=['DFM F1', 'DFM F2'],
                   columns=['PCA PC1', 'PCA PC2']).round(4))

# Visual comparison
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

for i, label in enumerate(['Factor 1', 'Factor 2']):
    ax = axes[i]
    # Align signs (flip PCA if correlation is negative)
    sign = np.sign(np.corrcoef(dfm_factors[:, i], pca_factors[:, i])[0, 1])
    pca_scaled = sign * pca_factors[:, i]
    # Standardize for visual comparison
    dfm_std = (dfm_factors[:, i] - dfm_factors[:, i].mean()) / dfm_factors[:, i].std()
    pca_std = (pca_scaled - pca_scaled.mean()) / pca_scaled.std()

    ax.plot(dates, dfm_std, 'b-', linewidth=1.5, label=f'DFM {label}')
    ax.plot(dates, pca_std, 'r--', linewidth=1.5, alpha=0.7, label=f'PCA {label}')
    ax.set_ylabel('Standardized Value')
    ax.set_title(f'{label}: DFM (Kalman Smoother) vs Static PCA')
    ax.legend()

plt.tight_layout()
plt.show()

## Comparison with statsmodels DynamicFactor

We compare kalmanbox's DFM with `statsmodels.tsa.statespace.DynamicFactor`
to validate the implementation.

In [ ]:
# statsmodels DynamicFactor comparison (K=2)
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor as SM_DFM

sm_model = SM_DFM(y, k_factors=2, factor_order=1)
sm_results = sm_model.fit(disp=False, maxiter=200)

print('statsmodels DynamicFactor summary:')
print(f'  Log-Likelihood: {sm_results.llf:.4f}')
print(f'  AIC: {sm_results.aic:.4f}')
print(f'  BIC: {sm_results.bic:.4f}')

# Compare log-likelihoods
print(f'\nkalmanbox  LogL: {results_k2.loglike:.4f}')
print(f'statsmodels LogL: {sm_results.llf:.4f}')

# Compare factors
# statsmodels factors.smoothed shape: (k_factors, nobs) -> transpose to (nobs, k_factors)
sm_factors = sm_results.factors.smoothed.T

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for i in range(2):
    ax = axes[i]
    kb_f = dfm_factors[:, i]
    sm_f = sm_factors[:, i]
    # Align sign
    sign = np.sign(np.corrcoef(kb_f, sm_f)[0, 1])
    sm_f_aligned = sign * sm_f
    # Standardize
    kb_std = (kb_f - kb_f.mean()) / kb_f.std()
    sm_std = (sm_f_aligned - sm_f_aligned.mean()) / sm_f_aligned.std()

    ax.plot(dates, kb_std, 'b-', linewidth=1.5, label='kalmanbox')
    ax.plot(dates, sm_std, 'r--', linewidth=1.5, alpha=0.7, label='statsmodels')
    corr = np.abs(np.corrcoef(kb_f, sm_f)[0, 1])
    ax.set_title(f'Factor {i+1}: kalmanbox vs statsmodels (|corr|={corr:.4f})')
    ax.set_ylabel('Standardized Factor')
    ax.legend()

plt.tight_layout()
plt.show()

# Comparison table
print('\nComparison summary:')
print(f'  |Correlation| Factor 1: {np.abs(np.corrcoef(dfm_factors[:, 0], sm_factors[:, 0])[0, 1]):.4f}')
print(f'  |Correlation| Factor 2: {np.abs(np.corrcoef(dfm_factors[:, 1], sm_factors[:, 1])[0, 1]):.4f}')

## Conclusions

1. **Single factor (K=1)**: Captures the dominant common variation across
   the macro panel — a broad "business cycle" factor.

2. **Two factors (K=2)**: Separates the panel into two latent dimensions,
   typically interpretable as "real activity" and "price/financial conditions."

3. **Factor loadings ($\Lambda$)**: The heatmap reveals which series load
   heavily on which factor, aiding economic interpretation.

4. **BIC selection**: The BIC criterion provides a data-driven way to choose
   $K$, balancing fit and parsimony.

5. **DFM vs PCA**: The DFM factors are highly correlated with PCA components,
   but the DFM additionally models temporal dynamics (factor persistence via
   $\Phi$) and produces proper probabilistic inference (filtered/smoothed
   estimates with uncertainty).

6. **Validation**: kalmanbox's DFM closely matches statsmodels' DynamicFactor,
   confirming correct implementation.

### Next Steps

- **Notebook 02**: Mixed-frequency DFM for nowcasting with missing data
- **Notebook 03**: Variance decomposition — how much of each series is
  explained by the common factors